# Git 本地基本操作

Git 把项目分成三个区域——工作区、暂存区、仓库（「办公桌 → 待归档文件篮 → 档案柜」）。本节就把这条流水线动手跑起来：跟踪文件、提交快照、查看历史与差异。这些操作在 [GitCode Fork 与 PR 流程](./06_gitcode_fork_pr.ipynb)（本课程的主线目标）中会反复用到。

你将学到：
1. 跟踪文件：`git add` 与 `git status`
2. 提交快照：`git commit`
3. 查看历史：`git log`
4. 查看差异：`git diff`
5. 撤销操作：`git checkout`、`git reset`
6. 用 `.gitignore` 忽略文件


---

## 1. 准备练习仓库

一个目录要变成 Git 仓库，只有两条路：

- **`git init`**：把当前目录变成一个全新的空仓库——仓库的“诞生”方式
- **`git clone`**：把已有仓库完整复制到本地——开源贡献的标准入口（见 [GitCode Fork 与 PR 流程](./06_gitcode_fork_pr.ipynb)）

无论哪种方式，目录下都会出现一个隐藏的 `.git` 目录，Git 的所有版本数据都存储在这里。

> ⚠️ 永远不要手动修改 `.git` 目录里的文件，除非你确切知道自己在做什么。

下面用 `git init` 从零创建一个练习仓库：


In [ ]:
# 环境准备：创建练习目录并初始化为 Git 仓库
import os, shutil, subprocess
demo = '/tmp/git_demo'
if os.path.exists(demo):
    shutil.rmtree(demo)
os.makedirs(demo)
os.chdir(demo)

!git init

# 检查提交身份（git config --global 配置过 user.name / user.email 即可）
# 若没有配置，这里为练习仓库补一个占位身份，避免后面 commit 报错
r = subprocess.run(['git', 'config', 'user.email'], capture_output=True, text=True)
if r.stdout.strip():
    print('已检测到提交身份:', r.stdout.strip())
else:
    !git config user.name "Learner"
    !git config user.email "learner@example.com"
    print('未检测到全局身份，已为练习仓库设置占位身份')

# 查看仓库状态
!git status


---

## 2. 跟踪文件：git add 与 git status

`git status` 是你最常用的「看一眼」命令——本节每个操作都配着它使用。

### 2.1 创建文件并查看状态


In [ ]:
# 创建一个 README 文件
with open("README.md", "w") as f:
    f.write("# My First Git Project\n\nHello Git!\n")

# 查看状态：README.md 显示为 "Untracked files"
!git status

### 2.2 git add：将文件加入暂存区

`git add` 把工作区的变更放入暂存区，告诉 Git “这些文件是我下一次提交要包含的”——也就是把要归档的文件放进「待归档文件篮」。


In [ ]:
# 将 README.md 加入暂存区
!git add README.md

# 再次查看状态：现在显示 "Changes to be committed"
!git status

---

## 3. 提交快照：git commit

`git commit` 把暂存区的内容正式登记入柜——「编号登记、归入档案柜」。

### 3.1 第一次提交


In [ ]:
# 提交暂存区的内容到仓库
!git commit -m "Initial commit: add README"

# 查看状态：working tree clean
!git status


### 3.2 提交信息的写法

好的提交信息是 Git 使用中最重要的习惯之一。推荐遵循 **Conventional Commits** 规范：

```
<type>(<scope>): <subject>

<body>
```

常见 type：

| type | 含义 |
|------|------|
| `feat` | 新功能 |
| `fix` | 修复 bug |
| `docs` | 文档变更 |
| `style` | 代码格式（不影响逻辑） |
| `refactor` | 重构 |
| `test` | 测试相关 |
| `chore` | 构建/工具/依赖等杂务 |

**示例：**
```bash
git commit -m "feat(login): 支持手机号验证码登录"
git commit -m "fix(parser): 修复空指针导致的崩溃"
git commit -m "docs: 更新 README 安装步骤"
```
**标注共同作者（Co-authored-by）**

两人搭档完成同一个提交（结对编程、他人指导、AI 辅助编码）时，在提交信息**正文末尾**加一行 `Co-authored-by` 尾注，平台会把这次提交同时计入两位作者的贡献：

```bash
git commit -m "feat(login): 支持手机号验证码登录" \
           -m "Co-authored-by: 张三 <zhangsan@example.com>"
```

要点：

- 格式固定：`Co-authored-by: 名字 <邮箱>`，名字与邮箱缺一不可
- 用**第二个 `-m`** 书写——它会成为提交信息的正文段落，尾注正好落在正文末尾
- 多位共同作者就写多行；尾注不影响标题，`git log --oneline` 依然干净


### 3.3 修改文件后再次提交

让我们修改 README，用 `git diff` 看看改了什么（详细用法见第 5 节），再走一遍完整的 add → commit 流程。


In [ ]:
# 修改 README
with open("README.md", "a") as f:
    f.write("\n## Features\n- Easy to use\n- Fast\n")

# 查看差异（工作区 vs 暂存区）
!git diff

In [ ]:
# 暂存并提交
!git add README.md
!git commit -m "docs: 添加 Features 章节"

!git status

---

## 4. 查看历史：git log

In [ ]:
# 查看提交历史（完整格式：作者、时间、提交信息）
!git log

# 单行格式——日常和 PR 流程中最常用
!git log --oneline


---

## 5. 查看差异：git diff

`git diff` 是理解“到底改了什么”的利器。

```bash
git diff                  # 工作区 vs 暂存区（还没 add 的改动）
git diff --staged         # 暂存区 vs 最新提交（已 add 但还没 commit 的改动）
```

> 💡 更多对比维度（`git diff HEAD`、提交之间、分支之间）见 [Git 指令汇总](./05_git_order.ipynb)。


---

## 6. 撤销操作

撤销是 Git 中最容易混淆的部分，下面是贡献流程中最常用的三个场景：

| 场景 | 命令 | 说明 |
|------|------|------|
| 改了工作区，还没 add | `git checkout -- <file>` | 丢弃工作区修改，恢复到暂存区状态 |
| 已经 add，还没 commit | `git reset HEAD <file>` | 撤出暂存区（文件修改保留） |
| 已经 commit，还没 push | `git reset --hard HEAD~1` | ⚠️ 彻底撤销提交和修改（不可恢复！） |

> 💡 更多撤销方式（`reset --soft`、`revert`、现代 `git restore` 语法）见 [Git 指令汇总](./05_git_order.ipynb)。


In [ ]:
# 示例：创建一个错误的修改，然后撤销
with open("README.md", "a") as f:
    f.write("\n这是误操作写入的内容\n")

print("--- 误修改后 ---")
!git diff

print("\n--- 撤销工作区修改 ---")
!git checkout -- README.md
!git status

> ⚠️ `git reset --hard` 会永久丢弃修改，使用前请三思。


---

## 7. 用 .gitignore 忽略文件

并非所有文件都应纳入版本控制：编译产物、日志、密钥、虚拟环境等应被忽略。

In [ ]:
# 创建一些不应被跟踪的文件
import os
os.makedirs("build", exist_ok=True)
with open("build/output.o", "w") as f: f.write("binary")
with open("debug.log", "w") as f: f.write("log")
with open(".env", "w") as f: f.write("SECRET=xxx")

# 创建 .gitignore
gitignore_content = """
# 编译产物
build/
*.o
*.so
*.exe

# 日志文件
*.log

# 环境变量 / 密钥（切勿提交！）
.env
*.pem
*.key

# Python 虚拟环境
__pycache__/
*.pyc
.venv/
venv/

# IDE 配置
.idea/
.vscode/
"""
with open(".gitignore", "w") as f: f.write(gitignore_content)

# 现在查看状态：build/、debug.log、.env 都被忽略，只有 .gitignore 待提交
!git status

### 7.1 .gitignore 的语法要点

```bash
*.log          # 忽略所有 .log 文件（* 是通配符）
build/         # 忽略整个 build 目录（末尾 / 表示目录）
.env           # 忽略指定文件——密钥类文件务必忽略
```

> 💡 **GitHub/gitcode 官方模板**：搜索 “gitignore template” 可以找到各语言的推荐 `.gitignore` 模板。


In [ ]:
# 提交 .gitignore
!git add .gitignore
!git commit -m "chore: 添加 .gitignore"

# 查看最终历史
!git log --oneline

---

## 8. 总结

**回顾本地工作流的核心循环：**

```bash
git status              # 1. 看看改了什么
git add <file>          # 2. 选择要提交的改动
git commit -m "msg"     # 3. 提交
git log --oneline       # 4. 查看历史
```

这正是「办公桌 → 文件篮 → 档案柜」的日常循环：`add` 把改动放进文件篮，`commit` 登记入柜，`status` / `log` / `diff` 让你随时看清各个区域的状态。

接下来，请学习 [分支管理](./03_git_branch.ipynb)，掌握 Git 最强大的分支能力！
